# t0211: Elsys tpc5 (ECR dual mode) → Labquake Explorer HDF5

Preprocessing for an experiment recorded entirely on the Elsys TranAX system.
Each run is one `.tpc5` file in ECR *dual* mode: **block 1** is the continuous
2 kHz record of the whole run, **blocks 2…N** are 2 MHz records around each PZT
trigger (0.1 s before, 0.5 s after), all on one clock.

The run's time history comes from block 1; the file itself is kept as a raw-data
reference (`run['elsys']`) so that **Extract Events** in the explorer reads the
2 MHz trigger block that contains each picked event.

Run sheet (2026-08-28, 21 °C, 58 % RH, flow rate 8 ml/min, operator 朱展穎):

| Elsys channel | sensor        | input range (Elsys setting) |
|---------------|---------------|-----------------------------|
| A1–A4         | PZT           | ±1 V (run1), ±0.5 V (run2–5), ±0.25 V (run6) |
| A5, A6        | pressure      | 0–10 V                      |
| A7            | LVDT          | −6 to 14 V                  |
| B1–B8         | eddy current  | ±25 V                       |

Runs: 01 8 MPa, 02 10 MPa, 03 12 MPa, 04 14 MPa, 05 16 MPa, 06 8 MPa.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib widget

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))   # repository root, if not pip-installed
from labquake_explorer.data.data_manager import DataManager
from labquake_explorer.data.sources import Tpc5Source, NINpzSource, open_source
from labquake_explorer.data.channels import get_channel, aligned_fields
from labquake_explorer.preprocessing import (
    Calibration, EddySlip, Friction, pressure_transducers, experiment,
    run_from_tpc5, run_from_tpc5_ni, slip_step_table,
)

## Paths

In [ ]:
EXPERIMENT = 't0211'
EXPERIMENT_DIR = Path('/Users/hueyke/Library/CloudStorage/SynologyDrive-data/Experiment/T0211')
DATA_DIR = EXPERIMENT_DIR / f'{EXPERIMENT}.exp' / 'data'

# Output read by Labquake Explorer; raw-file paths are stored relative to it, so keep it next to the data.
OUTPUT_FILE = EXPERIMENT_DIR / f'{EXPERIMENT}.h5'

RUN_FILES = sorted(DATA_DIR.glob(f'{EXPERIMENT}_*.tpc5'))
for p in RUN_FILES:
    print(f'{p.stat().st_size / 1e6:8.1f} MB  {p.name}')

## What is in a file

Channel and block tables of the first run (metadata only).

In [ ]:
src = Tpc5Source.open(RUN_FILES[0])
display(pd.DataFrame({'number': src.channel_numbers, 'name': src.channel_names}))
display(pd.DataFrame([b.as_dict() for b in src.blocks]))

## Channel roles and calibration

`CHANNELS` maps each Elsys channel label to the field stored in the run (volts).
`CALIBRATION` is the list of conversions applied on top; every step is also written
into `run['calibration']` so the file documents how its physical channels were made.

* Stresses: the lab's pressure-transducer calibration for a 1D fault with a 5 cm
  cross-section (Tseng 2026, NTU thesis, Appendix B.3.3 / Table B.1):
  `sigma_n = 4.507 V - 0.410` MPa on A5 and `V = 0.109 tau` on A6. The runs reproduce
  their nominal normal stress within 2 %.
* Slip: the stage calibration of the eddy-current sensors, `d (mm) = slope * V + intercept`,
  slopes -94.7, -96.6 and -95.4 um/V for the sensors on B1, B2, B3 (R² >= 0.9997, RMSE <= 10 um,
  travel up to 2 mm); the other five use the mean slope until they are calibrated. The jig
  intercepts are dropped and every `slip_k` is zeroed on the first 0.5 s of the run. The
  slope is negative (voltage falls as the gap opens), so slip accumulates positive.
  `displacement`, the field the Event Analyzer uses, is the mean of the eight slip channels
  by default (`EddySlip(displacement=...)` takes a single channel such as `'slip_5'` instead).
* The LVDT slope is still unknown; `lvdt` stays in volts.

`POSITIONS` places the sensors on the sample (mm; x along the fault from its left end,
y across the fault, z up). The table is stored with each recorder's `raw_data` and inherited by `slip`;
unknown entries stay NaN until filled in.

Layout of each run: `time`; one block per recorder (`elsys`, and `ni` for run5) holding the
raw-file reference and that recorder's voltages as the channel array `raw_data` (`data`
(channels x samples), `channels`, `unit`, `positions`); the physical scalars `normal_stress`,
`shear_stress`, `friction`, `displacement` at the top level; `slip` (channel array `slip_1..8`
in um with `source`, `slope_mm_per_v` and `positions`); `units`, `calibration`, `sources`.

`EVENT_FIELDS` / `EVENT_WINDOW_S` say which Elsys channels, and how much of each 2 MHz
trigger block, **Extract Events** copies into an event (the PZTs and the whole
0.1 s + 0.5 s block by default).

In [ ]:
CHANNELS = {          # Elsys channel -> field (volts)
    'A1': 'pzt_1',
    'A2': 'pzt_2',
    'A3': 'pzt_3',
    'A4': 'pzt_4',
    'A5': 'pressure_1',   # normal-stress transducer
    'A6': 'pressure_2',   # shear-stress transducer
    'A7': 'lvdt',
    'B1': 'eddy_1',
    'B2': 'eddy_2',
    'B3': 'eddy_3',
    'B4': 'eddy_4',
    'B5': 'eddy_5',
    'B6': 'eddy_6',
    'B7': 'eddy_7',
    'B8': 'eddy_8',
}

EDDY_MM_PER_V = {     # stage calibration, d (mm) = slope * V + intercept; sensors not listed use the mean slope
    'eddy_1': -0.094727676,
    'eddy_2': -0.096627292,
    'eddy_3': -0.095384964,
}

CALIBRATION = Calibration(
    *pressure_transducers('1D-5cm'),                                       # normal_stress, shear_stress (MPa)
    EddySlip(EDDY_MM_PER_V, zero_window_s=0.5, displacement='mean'),       # slip_1..8 and displacement (um)
    Friction(),
    note='pressure: Tseng 2026 Table B.1 (1D, 5 cm); eddy: stage calibration of B1-B3, mean slope elsewhere',
)

# Sensor positions on the sample (mm); None = unknown (stored as NaN). Fill in from the sensor layout.
POSITIONS = {         # field -> (x, y, z)
    'pzt_1': None,
    'pzt_2': None,
    'pzt_3': None,
    'pzt_4': None,
    'eddy_1': None,
    'eddy_2': None,
    'eddy_3': None,
    'eddy_4': None,
    'eddy_5': None,
    'eddy_6': None,
    'eddy_7': None,
    'eddy_8': None,
}
POSITION_FRAME = 'sample frame: x along the fault from its left end, y across the fault, z up'

EVENT_FIELDS = ['pzt_1', 'pzt_2', 'pzt_3', 'pzt_4']   # Elsys channels copied into events
EVENT_WINDOW_S = (0.1, 0.5)                             # seconds before / after the event

## Read the runs

In [ ]:
runs = [run_from_tpc5(p, CHANNELS, OUTPUT_FILE.parent, CALIBRATION,
                      event_fields=EVENT_FIELDS, event_window_s=EVENT_WINDOW_S,
                      positions=POSITIONS, position_frame=POSITION_FRAME)
        for p in RUN_FILES]
pd.DataFrame([{
    'name': r['name'], 'sigma_n nominal (MPa)': r['normal_stress_level'],
    'sigma_n measured (MPa)': float(np.nanmean(r['normal_stress'])),
    'samples': r['time'].size, 'duration (s)': float(r['time'][-1]),
    'trigger blocks': len(r['elsys']['blocks']['block']),
    'displacement end (um)': float(r['displacement'][-1]), 'start': r['start_time'][:19],
} for r in runs])

## Overview

Stresses and the mean slip of every run; shaded spans are the trigger blocks (where 2 MHz data exists).

In [ ]:
fig, axs = plt.subplots(len(runs), 1, figsize=(10, 2.2 * len(runs)), sharex=True)
for ax, run in zip(np.atleast_1d(axs), runs):
    ax.plot(run['time'], run['normal_stress'], lw=0.6, label='normal_stress (MPa)')
    ax.plot(run['time'], run['shear_stress'], lw=0.6, label='shear_stress (MPa)')
    ax2 = ax.twinx()
    ax2.plot(run['time'], run['displacement'], lw=0.6, color='C2')
    ax2.set_ylabel('displacement (um)', color='C2')
    blocks, dt0 = run['elsys']['blocks'], run['elsys']['time_offset']
    for a, b in zip(blocks['start'], blocks['end']):
        ax.axvspan(a + dt0, b + dt0, color='0.8', zorder=-10)
    ax.set_ylabel(f"{run['name']}  {run['normal_stress_level']:g} MPa" + ('  (NI)' if 'ni' in run else ''))
    ax.legend(loc='upper left', fontsize='small')
axs[-1].set_xlabel('time (s)')
fig.tight_layout()

## Clock check

One trigger block against the continuous record of the same channel, read back through the stored reference exactly as the explorer will.

In [ ]:
run_idx, block_idx, field = 0, 0, 'pzt_1'
run = runs[run_idx]
src = open_source(run['elsys'], OUTPUT_FILE.parent, run)
t_trig = src.trigger_times()[block_idx]
win = src.read_window(t_trig - 0.005, t_trig + 0.02, [field])
sel = (run['time'] >= win.time[0]) & (run['time'] <= win.time[-1])

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(win.time, win.data[0], lw=0.4, label=f'block {win.block} @ {win.sample_rate:g} Hz')
ax.plot(run['time'][sel], get_channel(run, field)[sel], 'o-', ms=3, lw=0.8, label='continuous @ 2 kHz')
ax.axvline(t_trig, color='k', ls=':', label='trigger')
ax.set_xlabel('time (s)'); ax.set_ylabel(f'{field} (V)'); ax.legend(loc='upper right')
ax.set_title(f"{run['name']}: trigger block {win.block} vs continuous record")

## Experiment record and save

In [ ]:
EXPERIMENT = experiment(
    't0211', runs,
    date='2026-08-28', operators=['朱展穎'], temperature_C=21.0, humidity_percent=58.0, flow_rate_ml_min=8.0,
    acquisition=('Elsys TranAX ECR dual mode: block 1 continuous at 2 kHz, trigger blocks at 2 MHz with '
                 '0.1 s pre- and 0.5 s post-trigger; A1-A4 PZT, A5-A6 pressure, A7 LVDT, B1-B8 eddy current'),
)

In [ ]:
dm = DataManager()
dm.data = EXPERIMENT
dm.save_file(OUTPUT_FILE)
print(f'{OUTPUT_FILE}  {OUTPUT_FILE.stat().st_size / 1e6:.1f} MB')

## Check the file the way the explorer loads it

In [ ]:
check = DataManager()
check.load_file(OUTPUT_FILE)
rows = []
for run in check.get_data('runs'):
    row = {'run': run['name'], 'samples': len(run['time']),
           'series': len(aligned_fields(run, len(run['time']), recurse=False)),
           'trigger blocks': len(run['elsys']['blocks']['block']),
           'displacement end (um)': float(run['displacement'][-1])}
    for key in run['sources']:
        row[f'{key} file found'] = (OUTPUT_FILE.parent / run[key]['filename']).exists()
    rows.append(row)
pd.DataFrame(rows)

## Next

Open `t0211.h5` in Labquake Explorer, right-click `runs/[i]/shear_stress` → **Pick Events**,
then `event_indices` → **Extract Events**: each event gets the run's channels around the
pick plus `elsys.original` (the PZT channels from the 2 MHz trigger block that contains
the event, if any).